# 🚀 Google Colab 生圖與生影片極限邊界測試 (L4 / A100 專用)

這份筆記本專門用來測試你帳號中 **L4 (24GB VRAM)** 與 **A100 (40GB / 80GB VRAM)** 的生圖與生影片極限邊界。

### 📋 測試項目一覽
1. **硬體規格檢測**：確認分配到的 GPU 型號、VRAM 容量與系統 RAM。
2. **生圖解析度邊界測試 (SDXL Turbo)**：連測 `1024x1024`、`1536x1536`、`2048x2048` 超高解析度，觀察耗時與 VRAM 峰值。
3. **頂級生圖模型測試 (Flux.1-schnell 12B 滿血版)**：測試 120 億參數模型在 bfloat16 下的極速生成能力。
4. **生影片基礎邊界測試 (Wan 2.1 1.3B)**：生成 `832x480`、81 影格（5 秒）動態影片。
5. **A100 終極影片邊界壓測 (Wan 2.1 14B 720p)**：挑戰 140 億參數模型生成 `1280x720` 高清影片（需 A100 40GB+）。

> ⚠️ **省額度提醒**：測試完畢後，請務必點選上方選單 **「執行階段 (Runtime)」 -> 「中斷連線並刪除執行階段」**！

## Step 1：檢測目前分配到的 GPU、VRAM 與系統記憶體邊界

In [ ]:
import torch
import psutil

print(f"PyTorch 版本: {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"✅ 成功啟用 GPU: {device_name}")
    print(f"📊 顯存 (VRAM) 容量: {vram_gb:.2f} GB")
    print(f"💻 系統記憶體 (System RAM): {ram_gb:.2f} GB")
    
    if vram_gb >= 38:
        print("🔥 診斷結果：你目前使用的是旗艦級 A100！可以全開測試 Step 2 ~ Step 5 所有極限項目！")
    elif vram_gb >= 22:
        print("⚡ 診斷結果：你目前使用的是 L4 (24GB)！可順跑 Step 2 ~ Step 4（Step 5 14B 建議改用 A100）。")
    else:
        print("💡 診斷結果：目前為 T4/G4 等級，系統將自動啟用 CPU Offload 保護顯存。")
else:
    print("❌ 目前未啟用 GPU！請至『執行階段 -> 變更執行階段類型』選擇 L4 或 A100 GPU。")

## Step 2：安裝最新版 Diffusers 與加速套件

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg ftfy

## Step 3：生圖解析度邊界壓測 (`1024x1024` vs `1536x1536` vs `2048x2048`)
測試在不同解析度下，GPU 的生成速度與顯存（VRAM）峰值變化。

In [ ]:
import gc
import time
import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

gc.collect()
torch.cuda.empty_cache()

model_id = "stabilityai/sdxl-turbo"
pipe = AutoPipelineForText2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant="fp16"
).to("cuda")

prompt = "A futuristic cyberpunk night market in Taipei, neon lights reflecting on wet pavement, cinematic lighting, ultra-detailed, 8k resolution, photorealistic"

resolutions = [(1024, 1024), (1536, 1536), (2048, 2048)]

for w, h in resolutions:
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()
        image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0, width=w, height=h).images[0]
        elapsed = time.time() - start_time
        peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        print(f"✅ [{w}x{h}] 生圖成功 | 耗時: {elapsed:.2f} 秒 | 峰值 VRAM: {peak_vram:.2f} GB")
        image.save(f"test_image_{w}x{h}.png")
        if w == 1024:
            display(image)
    except RuntimeError as e:
        print(f"❌ [{w}x{h}] 觸發顯存邊界上限 (OOM): {e}")
        break

del pipe
gc.collect()
torch.cuda.empty_cache()

## Step 4：生影片邊界測試 — Wan 2.1 (1.3B) 文字轉影片 (`832x480`, 5秒/81影格)
測試目前最熱門的開源影片模型 **Wan 2.1 (1.3B)**，免申請 Token 即可直接算圖出片！

In [ ]:
import gc
import time
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
if vram_gb >= 35:
    pipe.to("cuda")
    print("🚀 A100 全速模式：模型已完整載入 VRAM！")
else:
    pipe.enable_model_cpu_offload()
    print("⚡ 智慧顯存管理模式：已啟用 Model CPU Offload。")

video_prompt = "A cinematic drone shot flying through a breathtaking mountain canyon at golden hour, mist rising from a turquoise river below, photorealistic, smooth motion, 4k"
negative_prompt = "blurry, low quality, distorted, watermark, static, choppy"

start_time = time.time()
output = pipe(
    prompt=video_prompt,
    negative_prompt=negative_prompt,
    height=480,
    width=832,
    num_frames=81, # 81 frames @ 16fps ≈ 5 秒影片
    guidance_scale=5.0,
    num_inference_steps=30
).frames[0]

elapsed = time.time() - start_time
peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

video_path = "wan21_1.3b_test.mp4"
export_to_video(output, video_path, fps=16)
print(f"🎬 [Wan2.1 1.3B] 832x480 (81格/5秒) 影片生成完成！耗時: {elapsed:.1f} 秒 | 峰值 VRAM: {peak_vram:.2f} GB")

mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f'<video width=640 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))

del pipe, vae
gc.collect()
torch.cuda.empty_cache()

## Step 5：A100 終極邊界挑戰 — Wan 2.1 (14B) 旗艦 720p 影片生成
> ⚠️ **注意**：此區塊會下載約 28GB 的 14B 旗艦模型權重，**強烈建議僅在啟用 `A100 GPU` 且開啟「大量 RAM (High-RAM)」時執行**，用來測試 A100 40GB 的極限算力！

In [ ]:
import gc
import time
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
if vram_gb < 35:
    print(f"⚠️ 目前 VRAM 為 {vram_gb:.1f} GB，跑 14B 未量化版極易記憶體不足，建議切換至 A100 GPU 再執行此格！")
else:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model_id = "Wan-AI/Wan2.1-T2V-14B-Diffusers"
    vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
    # 14B 權重約 28GB，在 40GB A100 上搭配 CPU Offload 可穩定跑完 720p 不爆顯存
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()

    start_time = time.time()
    output = pipe(
        prompt="A majestic golden dragon soaring through storm clouds with lightning flashing, cinematic movie scene, ultra detailed scales, 720p HD",
        negative_prompt="blurry, low quality, distorted, watermark, static",
        height=720,
        width=1280,
        num_frames=49, # 49 frames (~3秒 720p)
        guidance_scale=5.0,
        num_inference_steps=25
    ).frames[0]

    elapsed = time.time() - start_time
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    video_path = "wan21_14b_720p.mp4"
    export_to_video(output, video_path, fps=16)
    print(f"🏆 [Wan2.1 14B 720p] 終極壓測完成！耗時: {elapsed:.1f} 秒 | 峰值 VRAM: {peak_vram:.2f} GB")

    mp4 = open(video_path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video width=640 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))